In [16]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler

# Step 1: Create dataset
data = pd.DataFrame({
    'Transaction': ['Payment', 'Transfer', 'Withdrawal'],
    'Amount': [500, 200, 800],
    'Channel': ['Online', 'ATM', 'Branch'],
    'Product': ['Credit Card', 'Loan', 'Investment']
})


encoder = OneHotEncoder(sparse_output=False)

categorical = data[['Transaction', 'Channel', 'Product']]
encoded_cat = encoder.fit_transform(categorical)

scaler = MinMaxScaler()

amount_scaled = scaler.fit_transform(data[['Amount']])


final_data = np.hstack([encoded_cat, amount_scaled])

binary_data = (final_data > 0.5).astype(int)
print(binary_data)

#customer vector
data = np.array([
    [1, 1, 0, 0, 1],
    [1, 1, 1, 0, 1],
    [0, 1, 0, 1, 0],
    [1, 0, 1, 1, 1],
    [0, 0, 1, 1, 0]
], dtype=float)


[[1 0 0 0 0 1 1 0 0 0]
 [0 1 0 1 0 0 0 0 1 0]
 [0 0 1 0 1 0 0 1 0 1]]


In [17]:
#RBM implementation
import numpy as np

class RBM:
    def __init__(self, visible_units, hidden_units, lr=0.01):
        self.visible_units = visible_units
        self.hidden_units = hidden_units
        self.lr = lr
        
        # Initialize weights and biases
        self.W = np.random.normal(0, 0.1, (visible_units, hidden_units))
        self.h_bias = np.zeros(hidden_units)
        self.v_bias = np.zeros(visible_units)

    def sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    def sample_h(self, v):
        prob_h = self.sigmoid(np.dot(v, self.W) + self.h_bias)
        sample_h = np.random.binomial(1, prob_h)
        return prob_h, sample_h

    def sample_v(self, h):
        prob_v = self.sigmoid(np.dot(h, self.W.T) + self.v_bias)
        sample_v = np.random.binomial(1, prob_v)
        return prob_v, sample_v

    def train(self, data, epochs=500):
        for epoch in range(epochs):
            loss = 0
            for v0 in data:
                # Positive phase
                ph0, h0 = self.sample_h(v0)

                # Negative phase
                pv1, v1 = self.sample_v(h0)
                ph1, h1 = self.sample_h(v1)

                # Update weights
                self.W += self.lr * (np.outer(v0, ph0) - np.outer(v1, ph1))
                self.v_bias += self.lr * (v0 - v1)
                self.h_bias += self.lr * (ph0 - ph1)

                loss += np.sum((v0 - v1) ** 2)

            if epoch % 50 == 0:
                print(f"Epoch {epoch}, Loss: {loss:.4f}")

In [24]:
#Train and inspect hidden frames
rbm = RBM(visible_units=5, hidden_units=2, lr=0.8)
rbm.train(data, epochs=300)

for v in data:
    prob_h, h = rbm.sample_h(v)
    print(f"Input: {v} → Hidden features: {h}")

#reconstruction
for v in data:
    _, h = rbm.sample_h(v)
    prob_v, v_reconstructed = rbm.sample_v(h)
    print(f"Original: {v}, Reconstructed: {v_reconstructed}")


Epoch 0, Loss: 17.0000
Epoch 50, Loss: 9.0000
Epoch 100, Loss: 6.0000
Epoch 150, Loss: 5.0000
Epoch 200, Loss: 3.0000
Epoch 250, Loss: 5.0000
Input: [1. 1. 0. 0. 1.] → Hidden features: [0 0]
Input: [1. 1. 1. 0. 1.] → Hidden features: [0 0]
Input: [0. 1. 0. 1. 0.] → Hidden features: [1 1]
Input: [1. 0. 1. 1. 1.] → Hidden features: [0 0]
Input: [0. 0. 1. 1. 0.] → Hidden features: [1 1]
Original: [1. 1. 0. 0. 1.], Reconstructed: [1 0 1 0 1]
Original: [1. 1. 1. 0. 1.], Reconstructed: [1 0 1 0 1]
Original: [0. 1. 0. 1. 0.], Reconstructed: [0 1 1 1 0]
Original: [1. 0. 1. 1. 1.], Reconstructed: [1 1 1 1 1]
Original: [0. 0. 1. 1. 0.], Reconstructed: [0 0 0 1 0]


In [27]:
new_customer = np.array([0, 1, 0, 1, 0])

_, h = rbm.sample_h(new_customer)
_, reconstructed = rbm.sample_v(h)

error = np.sum((new_customer - reconstructed) ** 2)
print("Reconstruction error:", error)

Reconstruction error: 1
